## NBA Exploratory Data Analysis
<img src = "NBA Logo.png" width = "400" height = "200">

*The National Basketball Association, or NBA, is a professional basketball league comprised of 30 teams across North America featuring the best basketball players in the world.* (NBA.com)


 In this project, I will be analyzing NBA Teams and their statistics to identify what separates NBA winners from losers. The core question I will be answering is: **Across the 2024-2025 season, which stats most strongly correlate with winning?** 


Data is sourced live from the NBA Stats API using the nba_api Python Library, covering all 30 teams across the 2024-2025 regular season.


We begin by importing significant modules, then fetching and exploring data from the API.

### Data Exploration

In [15]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import teamgamelog
import pandas as pd
import time

all_teams = teams.get_teams()
print(f"Total teams: {len(all_teams)}")

all_gamelogs = []

for team in all_teams:
    time.sleep(1)
    log = teamgamelog.TeamGameLog(team_id = team['id'], season='2024-25')
    df = log.get_data_frames()[0]
    df['team_name'] = team['full_name']
    all_gamelogs.append(df)

master_df = pd.concat(all_gamelogs, ignore_index=True)
print(f"\nDone! Shape: {master_df.shape}")

master_df.to_csv('nba_gamelogs.csv', index=False)


Total teams: 30

Done! Shape: (2460, 28)


In [16]:
master_df.head()

,Team_ID,Game_ID,GAME_DATE,MATCHUP,WL,W,L,W_PCT,MIN,FGM,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,team_name
0,1610612737,0022401186,"APR 13, 2025",ATL vs. ORL,W,40,42,0.488,240,47,...,9,35,44,32,8,2,15,15,117,Atlanta Hawks
1,1610612737,0022401173,"APR 11, 2025",ATL @ PHI,W,39,42,0.481,240,45,...,18,35,53,28,9,5,14,15,124,Atlanta Hawks
2,1610612737,0022401169,"APR 10, 2025",ATL @ BKN,W,38,42,0.475,240,48,...,7,44,51,36,11,1,15,19,133,Atlanta Hawks
3,1610612737,0022401149,"APR 08, 2025",ATL @ ORL,L,37,42,0.468,240,40,...,14,27,41,25,7,6,18,26,112,Atlanta Hawks
4,1610612737,0022401136,"APR 06, 2025",ATL vs. UTA,W,37,41,0.474,240,54,...,11,31,42,43,7,3,11,19,147,Atlanta Hawks


In [17]:
master_df.dtypes

Team_ID        int64
Game_ID          str
GAME_DATE        str
MATCHUP          str
WL               str
W              int64
L              int64
W_PCT        float64
MIN            int64
FGM            int64
FGA            int64
FG_PCT       float64
FG3M           int64
FG3A           int64
FG3_PCT      float64
FTM            int64
FTA            int64
FT_PCT       float64
OREB           int64
DREB           int64
REB            int64
AST            int64
STL            int64
BLK            int64
TOV            int64
PF             int64
PTS            int64
team_name        str
dtype: object

In [18]:
master_df.isnull().sum()

Team_ID      0
Game_ID      0
GAME_DATE    0
MATCHUP      0
WL           0
W            0
L            0
W_PCT        0
MIN          0
FGM          0
FGA          0
FG_PCT       0
FG3M         0
FG3A         0
FG3_PCT      0
FTM          0
FTA          0
FT_PCT       0
OREB         0
DREB         0
REB          0
AST          0
STL          0
BLK          0
TOV          0
PF           0
PTS          0
team_name    0
dtype: int64

**⭐️ Key Insights:** 


The dataset contains 2460 rows and 28 columns. Each column provides important information about each game:
- Identity columns: Team_ID, Game_ID, GAME_DATE, MATCHUP, team_name
- Outcome columns: WL (win or lose), W (total wins), L (total losses), W_PCT (win percentage so far)
- Box score columns: 
    - MIN                    (minutes played)
    - FGM, FGA, FG_PCT       (field goals made, attempted, percentage)
    - FG3M, FG3A, FG3_PCT    (three pointers made, attempted, percentage)
    - FTM, FTA, FT_PCT       (free throws made, attempted, percentage)
    - OREB, DREB, REB        (rebounds, offensive, defensive) 
    - AST                    (assists)
    - STL, BLK               (steals, blocks)
    - TOV                    (turnovers) 
    - PF                     (personal fouls)
    - PTS                    (points)

Fortunately, there are no missing values. Therefore, the data is ready for aggregation.

### Data Aggregation

Currently the dataset has 2,460 rows — one row per game per team. To compare teams against each other, we collapse this into 30 rows, one per team, containing season averages for each stat.